# 068 — Síntesis de voz y clonación responsable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Pipeline TTS:** texto → **normalización** (números, siglas, fechas) + **G2P** (letras →
fonemas) → **modelo acústico** (Tacotron 2: seq2seq con atención que predice el log-mel de
80 bandas, hop ~12,5 ms, más un *stop token*) → **vocoder** (mel → onda, porque el mel
descarta la fase).

**Vocoders:** WaveNet (2016) genera muestra a muestra con convoluciones causales dilatadas
(`p(x) = ∏ p(xₜ|x₁…xₜ₋₁)`): calidad casi humana, pero 16 000 pasos por segundo de audio.
Los vocoders paralelos (HiFi-GAN) generan toda la onda en una pasada → tiempo real.

**Clonación:** un *speaker encoder* comprime segundos de voz en un d-vector que condiciona
el modelo acústico (SV2TTS). **Responsabilidad:** consentimiento explícito y documentado,
watermarking imperceptible (degradable por re-grabación/compresión) y detección de audio
sintético. **Evaluación:** MOS (1-5, subjetivo, reportar media *y* desvío) y WER de un ASR
sobre el audio sintético como proxy de inteligibilidad.


### 🧮 Cálculo de referencia (para los ejercicios)

```text
Campo receptivo WaveNet = 1 + Σ dilaciones
10 capas kernel 2, dilaciones 1,2,4,…,512 → 1 + 1023 = 1024 muestras = 64 ms a 16 kHz

MOS: A=[4,4,5,3,4] → media 4.0, desvío 0.63 | B=[5,5,5,1,4] → media 4.0, desvío 1.55
```


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=68)
show(result)


## Reflexión

1. Una familia pide clonar la voz de un pariente fallecido para un homenaje. ¿Quién puede
   consentir en ese caso, y qué límites de alcance y revocación pondrías por escrito?
2. El watermark de tu TTS sobrevive a la compresión MP3 pero no a la re-grabación con un
   micrófono. ¿Sigue siendo útil? ¿Dentro de qué estrategia de defensa más amplia?
3. Tu TTS lee recetas médicas en voz alta (clase 072). ¿Qué error del frontend de
   normalización sería el más peligroso ("500 mg", "c/8 h") y cómo lo detectarías antes de
   desplegar?
